In this notebook, I classify the videos into categories based on their title.

Create the connection to the database.

In [1]:
import sqlite3
import pandas as pd 

conn = sqlite3.connect('../data/lafc_content.db')

Open the stored SQL query with combined video vs lafc match context table, and pull it into the base_df dataframe.

In [2]:
with open('../sql/videos_vs_lafc_match_context.sql') as f:
    query = f.read()
base_df = pd.read_sql(query, conn)

base_df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,season,kickoff_utc,...,home_away,goals_for,goals_against,lafc_points,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match
0,5o8eC2o3ns4,Inside LAFC | Episode 214 - Leagues Cup,Max Bretos & Benny Feilhaber pull back the cur...,2026-08-11T14:24:24Z,PT53M21S,1536,86,4,2026,2026-08-01T23:30:00Z,...,A,1,1,33,18,10,33,16,10,9.62
1,XL7Xk-133mM,The Team On Eddie's Winner | TOL vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-09T10:00:34Z,PT40S,1986,65,17,2026,2026-08-01T23:30:00Z,...,A,1,1,33,18,10,33,16,10,7.44
2,YTs2d9hkg-Y,Eddie's Goal vs TOL | EVERY ANGLE,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-09T09:15:16Z,PT46S,3307,135,23,2026,2026-08-01T23:30:00Z,...,A,1,1,33,18,10,33,16,10,7.41
3,YBaIh-Hs534,TOL vs LAFC | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-09T08:45:20Z,PT14M12S,1220,36,35,2026,2026-08-01T23:30:00Z,...,A,1,1,33,18,10,33,16,10,7.39
4,o7jY-G-M8Mw,Eddie Called Game 😤,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-09T08:30:34Z,PT40S,5913,243,23,2026,2026-08-01T23:30:00Z,...,A,1,1,33,18,10,33,16,10,7.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3546,EZP7m0-qpGA,"""We can't wait."" | Vela On Banc of California ...",,2018-03-06T20:17:55Z,PT1M1S,12628,170,10,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,1.93
3547,SME_lvoBvuw,WATCH: A closer look at the first goal in LAFC...,,2018-03-06T20:17:53Z,PT30S,1240,28,1,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,1.93
3548,aAeIq6bAdTI,All-Access: Behind the Scenes of LAFC's First ...,"An exclusive, behind the scenes look at the fi...",2018-03-05T20:14:02Z,PT3M6S,6608,193,14,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,0.93
3549,ff8VZHNeD9k,Diego Rossi Scores The First Goal in LAFC Hist...,Diego Rossi scored a beautiful goal in LAFC's ...,2018-03-05T02:52:36Z,PT1M9S,13911,250,18,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,0.20


Import Tfid and kMeans from scikit learn.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

Create a series of titles from the base data frame, I'm ignoring the description because it's repeated across different types of videos, and biases clustering.

In [4]:
text = base_df['title'].fillna('')
text.head()

0     Inside LAFC | Episode 214 - Leagues Cup
1    The Team On Eddie's Winner | TOL vs LAFC
2           Eddie's Goal vs TOL | EVERY ANGLE
3               TOL vs LAFC | Postmatch Media
4                         Eddie Called Game 😤
Name: title, dtype: str

Adding (too) commonly occuring title words to the English stop words list:

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
custom = ENGLISH_STOP_WORDS.union({'lafc','https','com','www','la','los','angeles','ep','episode'})

Vectorize the text series, into a sparse matrix, using tfid, and load it in a feature variable X.

In [6]:
vec = TfidfVectorizer(
    stop_words=list(custom),   # use my custom words as stop_words
    ngram_range=(1, 2),     # single words AND 2-word phrases ("inside lafc")
    min_df=5,               # ignore terms that appear in fewer than 5 videos (rare noise)
    max_df=0.4,
    max_features=500,       # cap vocabulary size — keeps it fast/focused
)
X = vec.fit_transform(text)

Create the kmeans model and assign each video a cluster.

In [7]:
k = 8
km = KMeans(n_clusters=k, random_state=11, n_init=10)
base_df['cluster'] = km.fit_predict(X)

Examining the clusters:

In [8]:
terms = vec.get_feature_names_out()
for c in range(k):
    top_idx = km.cluster_centers_[c].argsort()[-10:][::-1]   # 10 highest-weight terms
    keywords = ", ".join(terms[i] for i in top_idx)
    examples = base_df[base_df.cluster == c]['title'].head(3).tolist()
    print(f"\ncluster {c} (n={(base_df.cluster==c).sum()})")
    print(f"  keywords: {keywords}")
    print(f"  examples: {examples}")


cluster 0 (n=36)
  keywords: pitch, presented waymo, waymo, pitch presented, presented, hollingshead, son, ryan hollingshead, ryan, son heung
  examples: ['Steve Cherundolo | Off The Pitch | Presented by Waymo', 'Off The Pitch | Ryan Hollingshead | Presented by Waymo', 'Yaw Yeboah | Off The Pitch | Presented by Waymo']

cluster 1 (n=39)
  keywords: diego, rossi, diego rossi, goal diego, goal, united, history, anatomy goal, anatomy, scores
  examples: ["Academy Report | Diego's Journey", 'Part of our History | Diego "Chiqui" Palacios', 'LAFC x 110 Football | Thanks For The Memories Diego Rossi & Rayito Strikes Twice']

cluster 2 (n=167)
  keywords: inside, inside podcast, podcast, max vince, vince, cup, max, inside max, john, home
  examples: ['Inside LAFC | Episode 214 - Leagues Cup', 'Inside LAFC | Episode 213 - On to Leagues Cup', 'Inside LAFC | Episode 212 - Off to a hot start']

cluster 3 (n=215)
  keywords: gold, black gold, black, gold insider, insider, gold weekly, weekly, pres

Based on the clustered keywords, I created a rules based classify function. I also created a format family dictionary, to group related cateogries, post function.

In [9]:
def classify_format(title):
    """
    Rule-based format classifier for LAFC videos.
    Order matters: named series first, then press/interview, then match content,
    then themed content, catch-all last. First match wins. Title-only, no regex.
    """
    t = str(title).lower()

    # 1. Named recurring series
    if 'inside lafc' in t:                                          return 'inside_lafc'
    if 'gold insider' in t:                                         return 'black_and_gold'   # the SHOW only
    if 'is black & gold' in t or 'is black and gold' in t:          return 'signing'          # "X is Black & Gold"
    if 'mvp podcast' in t:                                          return 'mvp_podcast'
    if 'acción' in t or 'accion' in t:                             return 'accion_lafc'
    if 'lafc weekly' in t or 'weekly | ' in t:                     return 'lafc_weekly'
    if 'lafc+' in t or 'lafc +' in t:                              return 'lafc_plus'
    if 'on the mic' in t:                                           return 'on_the_mic'
    if 'behind the crest' in t:                                     return 'behind_the_crest'
    if 'away days' in t:                                            return 'away_days'
    if 'a lot more to prove' in t:                                  return 'more_to_prove'
    if 'staying home' in t:                                         return 'staying_home'
    if '안녕 lafc' in t:                                            return 'korean_series'
    if 'podcast' in t:                                              return 'other_podcast'

    # 2. Press / interview (keyword-based)
    if 'postmatch' in t or 'post-match' in t:                      return 'postmatch_media'
    if 'prematch' in t or 'pre-match' in t:                        return 'prematch_media'
    if 'conference' in t or 'presser' in t or 'media availab' in t:  # 'availab' catches the typo
        return 'presser'
    if any(w in t for w in ['in touch','speaks','discusses','talks ',
                            'thoughts','reacts','reaction','breaks down','sits down']):
        return 'interview'

    # 3. Match content
    if 'highlight' in t or 'all goals' in t or 'every goal' in t or 'goal scored' in t \
       or 'top ten goals' in t or 'top 10 goals' in t or 'best goals' in t or 'top goals' in t:
        return 'highlights'
    if ('goal:' in t or t.startswith('goal ') or t.startswith('goal!')
            or '| goal' in t or 'every angle' in t or 'wondergoal' in t or 'golazo' in t
            or 'game winner' in t or 'from the spot' in t or 'from pitchside' in t):
        return 'goal_clip'
    if 'save of the match' in t or 'huge save' in t or 'leaping save' in t or 'big save' in t:
        return 'save_clip'
    if 'match frames' in t:                                         return 'match_frames'
    if 'preview' in t or 'keys to the match' in t:                 return 'match_preview'
    if 'recap' in t:                                                return 'recap'

    # 4. Themed / behind-the-scenes / features
    if 'behind the scenes' in t or 'sounds of' in t:               return 'behind_the_scenes'
    if 'training' in t or "mic'd up" in t or 'micd up' in t:       return 'training'
    if 'watch party' in t or 'watch along' in t or 'watch-along' in t or '110 football' in t:
        return 'watch_party'
    if 'built for it' in t:                                         return 'built_for_it'
    if any(w in t for w in ['get to know','player profile','lafc profile','join the club',
                            'on this day','cali to cali']):         return 'feature'

    # 5. Catch-all
    return 'unclassified'


FORMAT_FAMILY = {
    # produced recurring series
    'inside_lafc':      'show',
    'black_and_gold':   'show',
    'mvp_podcast':      'show',
    'accion_lafc':      'show',
    'lafc_weekly':      'show',
    'lafc_plus':        'show',
    'on_the_mic':       'show',
    'behind_the_crest': 'show',
    'away_days':        'show',
    'more_to_prove':    'show',
    'staying_home':     'show',
    'korean_series':    'show',
    'other_podcast':    'show',

    # match content
    'highlights':       'match',
    'goal_clip':        'match',
    'save_clip':        'match',
    'match_frames':     'match',
    'match_preview':    'match',
    'recap':            'match',

    # press / interview
    'postmatch_media':  'media',
    'prematch_media':   'media',
    'presser':          'media',
    'interview':        'media',

    # themed / features
    'behind_the_scenes':'behind_scenes',
    'training':         'behind_scenes',
    'watch_party':      'watch_party',
    'built_for_it':     'feature',
    'feature':          'feature',
    'signing':          'signing',

    # catch-all
    'unclassified':     'unclassified', 
}

Apply the classify function, and map the format family, to two new columns in the base_df, then count them:

In [10]:
base_df['content_type']   = base_df['title'].apply(classify_format)
base_df['format_family']  = base_df['content_type'].map(FORMAT_FAMILY)

base_df['content_type'].value_counts()

content_type
unclassified         1866
highlights            246
goal_clip             174
inside_lafc           167
lafc_weekly           113
lafc_plus             106
accion_lafc            87
mvp_podcast            82
match_preview          66
prematch_media         65
recap                  65
signing                62
postmatch_media        60
black_and_gold         58
interview              53
behind_the_crest       49
feature                47
watch_party            39
training               36
presser                34
behind_the_scenes      21
korean_series          15
match_frames           11
away_days               9
on_the_mic              8
save_clip               4
staying_home            3
other_podcast           2
built_for_it            2
more_to_prove           1
Name: count, dtype: int64

Even with all my rules, many of the videos are not easily classifiable by title, those are assigned the category of "unclassified".

In [11]:
base_df[['title', 'format_family']].sample(30)

,title,format_family
833,World's City To The World Stage.,unclassified
1553,Bouanga: The Best Performance Of Tonight Is Th...,unclassified
2171,Highlights | LAFC vs. Galaxy (Western Conferen...,media
2523,LAFC Supporters Donate PPE To LA Street Vendors,unclassified
187,LAFC vs. Colorado Rapids | MATCH HIGHLIGHTS,match
2525,Kim Moon-Hwan Is Black & Gold,signing
2585,A Look Behind The Scenes With Equipment Manage...,behind_scenes
1220,Simply the Best | #Best XI,unclassified
3313,From The UK With Love | UKLAFC,unclassified
135,Martínez connects with Bouanga against Nashville,unclassified


I tuned the classification based on rules as best I could, but I still had ≈ 1800 videos that I was labeling "unclassified" they were mostly short social clips without a standard title, but some were interview clips, so I decided to try a light machine learning model on the "unclassified" set to see if I could label those more finely. 

In [12]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/laptop_02/Documents/data_projects/lafc_content/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11575.61it/s]


In [13]:
CATEGORIES = {
    'goal_clip':   'a video clip of a single goal being scored in a match',
    'highlights':  'match highlights or a compilation of multiple goals',
    'interview':   'a player or coach speaking, quote, press interview or reaction',
    'feature':     'a player profile, personal story, or getting-to-know feature',
    'behind_the_scenes': 'behind the scenes footage, training, or documentary content',
    'match_preview':'a preview or build-up before an upcoming match',
    'recap':       'a season or match recap looking back',
    'misc_social_clip': 'a short hype or social media clip, slogan, or promotional post',
}

In [14]:
cat_names = list(CATEGORIES.keys())
cat_embeddings = model.encode(list(CATEGORIES.values()))

In [15]:
unclassified = base_df[base_df['content_type'] == 'unclassified'].copy()
title_embeddings = model.encode(unclassified['title'].fillna('').tolist(), show_progress_bar=True)

Batches: 100%|██████████| 59/59 [00:01<00:00, 46.29it/s]


In [16]:
import numpy as np
sims = util.cos_sim(title_embeddings, cat_embeddings).numpy()   # shape: (n_titles, n_categories)
best_idx   = sims.argmax(axis=1)     # which category scored highest per title
best_score = sims.max(axis=1)        # how strong that match was
unclassified['ml_label'] = [cat_names[i] for i in best_idx]
unclassified['ml_score'] = best_score

In [17]:
THRESHOLD = 0.30   # tune this — see below
unclassified['ml_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'          # weak match -> genuine miscellany
)

In [18]:
unclassified[['title', 'ml_label', 'ml_score']].sample(30)

,title,ml_label,ml_score
1576,🗣️ Happy for my first assist! | Hugo Lloris wi...,misc_social_clip,0.259671
871,All to play for.,misc_social_clip,0.251058
1389,🗣️ HEY! HEY! HEY! | #LAFC #MLS #OpenCup #Llori...,highlights,0.317319
3236,Best Of 2018 | Assists,highlights,0.303137
3437,Diomande: 'It’s Just A Bonus To Get That Histo...,misc_social_clip,0.212153
406,Playoff focus ⚡️,highlights,0.317724
585,Near post finish by Bouanga 😮‍💨,recap,0.306142
3510,WHAT A MATCH: LAFC vs. Montreal Impact | April...,goal_clip,0.472115
1303,Playoff Mentality: Do or Die.,misc_social_clip,0.263516
2465,LAFC & Eduard Atuesta Agree To A Contract Exte...,misc_social_clip,0.095822


In [19]:
unclassified['ml_score'].describe()

count    1866.000000
mean        0.249485
std         0.094298
min         0.024242
25%         0.182427
50%         0.246149
75%         0.310362
max         0.584862
Name: ml_score, dtype: float64

In [20]:
unclassified['dur_min'] = pd.to_timedelta(unclassified['duration']).dt.total_seconds() / 60

u = unclassified.sort_values('ml_score', ascending=False)

print("=== HIGH scores (top matches) ===")
print(u.head(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== MID scores (around the median) ===")
print(u.iloc[900:915][['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== LOW scores (weakest) ===")
print(u.tail(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

=== HIGH scores (top matches) ===
                                                                         title           ml_label  ml_score  dur_min
3547                    WATCH: A closer look at the first goal in LAFC history          goal_clip     0.585      0.5
473                                             ✌️ Regular season matches left              recap     0.570      0.3
782                                    Goals in back-to-back matches for Ordaz         highlights     0.527      0.5
921                                                     4️⃣ goals in one place         highlights     0.526      0.6
2950                       11 Goals Over In Our Last 2 Games | Watch Them All!         highlights     0.524      1.0
6                                                                      SCENES.  behind_the_scenes     0.520      0.3
1514                    1 Goal = 3 Celebrations | #LAFC #MLS #LeaguesCup #Goal         highlights     0.518      0.2
2985                Anatomy Of

The light ml model did well at recognizing goal and highlight clips, but it was missing interview clips. I decide to provide the model with category vectors based on actual examples titles for each category, instead of providing it with text descriptions of the categories, to see if that would improve its classification.

In [21]:
PROTOTYPES = {
    'goal_clip': [
        # structured (with score line)
        "GOAL: M. Bogusz vs VAN, 1'",
        "Denis Bouanga breaks the tie! LAFC 2 - 1 HOU",
        "Diego Rossi opens the scoring, LAFC 1 - 0 Dallas",
        # short descriptive goal moments (no score line — the leaking shape)
        "Sonny picks his spot 🎯",
        "Near post finish by Bouanga 😮‍💨",
        "Bouanga chips the keeper | ALL ANGLES",
        "SONNY FROM DISTANCE 🚀",
        "Timmy's strike from the top of the box",
    ],

    'highlights': [
        "Highlights | LAFC vs FC Dallas",
        "Full Highlights | 3-0 | LAFC vs. Colorado Rapids",
        "MATCH HIGHLIGHTS | LAFC vs Seattle Sounders",
        "Every Goal From LAFC's Inaugural MLS Season",
        "11 Goals Over In Our Last 2 Games | Watch Them All",
    ],
    'interview': [
        "Cherundolo: We'll Need Effort Again Against Austin",
        "Ebobisse: Feeling more and more confident by the day",
        "Bradley Addresses Media After Mark-Anthony Kaye Trade",
        "Bouanga speaks on his hat trick",
        "Nguyen: This Is Where You Start To Play For Playoff Positions",
        "State Of The Union | Tom Penn",
        "Hollingshead: Huge Result For Us, Three Points On The Road",
    ],
    'feature': [
        "Get To Know Kwadwo Opoku",
        "LAFC Profile | From Norway to LA, Adama Diomande",
        "Building A Legacy | Carlos Vela's Past & Future With LAFC",
        "Join The Club | Juan Pinto",
        "The Call-Up | Christian Ramirez",
    ],
    'behind_the_scenes': [
        "Behind The Scenes | 2026 Primary Kit Shoot",
        "Sounds of Training | First Week Back",
        "A Look Behind The Scenes With Equipment Manager Scott Tranilla",
        "Inside the locker room after the win",
    ],
    'match_preview': [
        "LAFC at LA Galaxy - Match Preview",
        "Keys To The Match | LAFC vs Seattle",
        "Previewing the road trip to Colorado",
        "What to watch for ahead of LAFC vs Austin FC",
    ],
    'recap': [
        "2023 LAFC Season Recap",
        "Recap | LAFC vs Colorado Rapids",
        "Looking Back At The 2022 MLS Cup Run",
        "Year In Review | 2021 Season",
    ],
    'misc_social_clip': [
        "24 Hours ⏳",
        "The dagger 🗡️",
        "99 is electric ⚡️",
        "Ready to fight for the Club 🫡",
        "Pressure is a privilege",
        "It's about the collective.",
    ],
}

In [22]:
import numpy as np

cat_names = list(PROTOTYPES.keys())
cat_vectors = []
for name in cat_names:
    ex_embs = model.encode(PROTOTYPES[name])   # embed that category's example titles
    cat_vectors.append(ex_embs.mean(axis=0))   # average -> one "centroid" per category
cat_vectors = np.vstack(cat_vectors)

In [23]:
from sentence_transformers import util

sims = util.cos_sim(title_embeddings, cat_vectors).numpy()
best_idx = sims.argmax(axis=1)
unclassified['ml_label'] = [cat_names[i] for i in best_idx]
unclassified['ml_score'] = sims.max(axis=1)

In [24]:
THRESHOLD = 0.47   # tuned based on running the model a few times at looking at resulting ml scores against titles
unclassified['ml_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'
)

In [25]:
unclassified[['title', 'ml_label', 'ml_score']].sample(30)

,title,ml_label,ml_score
428,SONNY SCORES IN THE PLAYOFFS,goal_clip,0.531174
626,Son Heung Min's First MLS Goal | SIDELINE POV,goal_clip,0.530832
2157,Gareth Bale's MLS Cup Stunner As Heard Around ...,misc_social_clip,0.400709
2156,Maxime Crépeau. LAFC Legend.,misc_social_clip,0.404239
2656,LAFC Launches The Black & Gold Community Relie...,misc_social_clip,0.324737
2976,NFL Legend Shawne Merriman Visits The North End!,misc_social_clip,0.369587
1753,Cherundolo: We're Very Pleased With How We're ...,misc_social_clip,0.450052
2764,LAFC Goal Rush | Top 5 Plays in April – 2019,highlights,0.651542
1374,99's Aesthetic ✨ | #LAFC #MLS #DenisBouanga #F...,highlights,0.500360
2071,Maxime Crépeau | Road To Recovery Episode 2,misc_social_clip,0.305598


In [26]:
unclassified['ml_score'].describe()

count    1866.000000
mean        0.444876
std         0.111380
min         0.040219
25%         0.369217
50%         0.447446
75%         0.518103
max         0.823427
Name: ml_score, dtype: float64

In [27]:
unclassified['dur_min'] = pd.to_timedelta(unclassified['duration']).dt.total_seconds() / 60

u = unclassified.sort_values('ml_score', ascending=False)

print("=== HIGH scores (top matches) ===")
print(u.head(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== MID scores (around the median) ===")
print(u.iloc[900:915][['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== LOW scores (weakest) ===")
print(u.tail(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

=== HIGH scores (top matches) ===
                                                                                       title       ml_label  ml_score  dur_min
378                                                    A Year In Review | LAFC's 2025 Season          recap     0.823      4.6
3220                                             MEMORABLE MATCHDAY | LAFC Makes MLS History     highlights     0.768      3.9
2662                                               LAFC In 30 | LAFC vs. FC Dallas - 5/19/19     highlights     0.737     30.0
3522                                     WATCH: All 3 Goals in LAFC's 3-4 Loss vs. LA Galaxy     highlights     0.736      2.2
3510                                 WHAT A MATCH: LAFC vs. Montreal Impact | April 21, 2018     highlights     0.735      4.0
3547                                  WATCH: A closer look at the first goal in LAFC history     highlights     0.733      0.5
1784                                                       2024 Preseason: LA

It did a better job at classifying, and I went back and adjusted the ml_score threshold (below that the model would default a row to misc_social_clip). Then I merged the data frame of previously unclassified, back into the base_df, adding a content_type_final_column.

In [28]:
import numpy as np

# 1. Apply the threshold -> final label for the unclassified rows
THRESHOLD = 0.47
unclassified['final_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'
)

# FORMAT_FAMILY was written before the ML pass existed, so it has no entry
# for the one category only the model can produce. Add it here rather than
# editing the original dict, to keep the order of discovery visible.
FORMAT_FAMILY['misc_social_clip'] = 'social'

# 2. Start the merged column as the RULE labels (the trustworthy core)
base_df['content_type_final'] = base_df['content_type']

# 3. Overwrite ONLY the previously-unclassified rows with the ML result.
#    Aligns by index — works because `unclassified` kept base_df's index.
base_df.loc[unclassified.index, 'content_type_final'] = unclassified['final_label']

# 4. Re-map the coarse family on the merged labels
base_df['format_family'] = base_df['content_type_final'].map(FORMAT_FAMILY)

# 5. Sanity checks
print(base_df['content_type_final'].value_counts(), "\n")
print("misc_social_clip:", f"{(base_df.content_type_final=='misc_social_clip').mean():.0%}")
print("unmapped families (must be 0):", base_df['format_family'].isna().sum())

content_type_final
misc_social_clip     1154
highlights            454
goal_clip             385
inside_lafc           167
recap                 154
match_preview         143
feature               118
lafc_weekly           113
interview             109
lafc_plus             106
accion_lafc            87
mvp_podcast            82
prematch_media         65
signing                62
postmatch_media        60
black_and_gold         58
behind_the_crest       49
watch_party            39
training               36
presser                34
behind_the_scenes      21
korean_series          15
match_frames           11
away_days               9
on_the_mic              8
save_clip               4
staying_home            3
other_podcast           2
built_for_it            2
more_to_prove           1
Name: count, dtype: int64 

misc_social_clip: 32%
unmapped families (must be 0): 0


In [29]:
print(base_df['format_family'].value_counts(), '\n')

format_family
social           1154
match            1151
show              700
media             268
feature           120
signing            62
behind_scenes      57
watch_party        39
Name: count, dtype: int64 

